# SofaScore Upcoming Bundesliga Matches Extractor

This notebook retrieves upcoming SofaScore events for every team in the shared Bundesliga reference. It records the earliest valid upcoming event overall and the fixture belonging to the matchday after the user-entered squad-analysis matchday. The latter is accepted only when its SofaScore event ID occurs in the project match-ID file for that following matchday.

A single reusable undetected Chrome instance opens the API JSON responses and reads them from their rendered `<pre>` element. The browser is closed in a `finally` block even when individual teams or the overall run encounter errors.

## 1. Project paths

Locate the repository from either VS Code's notebook location or the kernel working directory, then import the authoritative output locations.

In [1]:
import sys
from pathlib import Path


def _locate_project_root() -> Path:
    starts = []
    vscode_notebook = globals().get('__vsc_ipynb_file__')
    if isinstance(vscode_notebook, str) and vscode_notebook.strip():
        starts.append(Path(vscode_notebook).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())

    checked = set()
    for start in starts:
        for candidate in (start, *start.parents):
            if candidate in checked:
                continue
            checked.add(candidate)
            if (candidate / 'project_paths.py').is_file():
                return candidate
    raise FileNotFoundError(
        'Could not locate project_paths.py. Start Jupyter from the Kickbase '
        'project root or open this notebook from within that project.'
    )


_PROJECT_ROOT = _locate_project_root()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from project_paths import (
    SOFASCORE_MATCH_IDS_DIR,
    SOFASCORE_REFERENCE_DIR,
    SOFASCORE_UPCOMING_MATCHES_DIR,
    ensure_directory,
)


## 2. Imports and run configuration

Set browser and pagination limits, request the Bundesliga matchday being analysed, derive its following matchday, and prepare a timezone-aware export location.

In [2]:
import json
from datetime import datetime, timezone
from typing import Any

try:
    import pandas as pd
    import undetected_chromedriver as uc
    from bs4 import BeautifulSoup
    from IPython.display import display
    from selenium.common.exceptions import TimeoutException, WebDriverException
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.support.ui import WebDriverWait
except ImportError as exc:
    raise ImportError(
        'Required packages are missing. Install undetected-chromedriver, selenium, '
        'beautifulsoup4, and pandas in this Jupyter kernel, then restart the kernel.'
    ) from exc

CHROME_MAJOR_VERSION = 150
HEADLESS = False
PAGE_LOAD_TIMEOUT_SECONDS = 30
WAIT_TIMEOUT_SECONDS = 20
MAX_PAGES = 10
MIN_ANALYSIS_MATCHDAY = 1
MAX_ANALYSIS_MATCHDAY = 34
BUNDESLIGA_UNIQUE_TOURNAMENT_ID = 35
TEAM_EVENTS_URL_TEMPLATE = (
    'https://www.sofascore.com/api/v1/team/{team_id}/events/next/{page}'
)

for setting_name, setting_value in {
    'CHROME_MAJOR_VERSION': CHROME_MAJOR_VERSION,
    'PAGE_LOAD_TIMEOUT_SECONDS': PAGE_LOAD_TIMEOUT_SECONDS,
    'WAIT_TIMEOUT_SECONDS': WAIT_TIMEOUT_SECONDS,
    'MAX_PAGES': MAX_PAGES,
}.items():
    if not isinstance(setting_value, int) or isinstance(setting_value, bool) or setting_value < 1:
        raise ValueError(f'{setting_name} must be a positive integer.')

try:
    analysis_matchday = int(input('What Bundesliga matchday are you analysing your squad for? '))
except ValueError as exc:
    raise ValueError('analysis_matchday must be an integer from 1 to 34.') from exc
if not MIN_ANALYSIS_MATCHDAY <= analysis_matchday <= MAX_ANALYSIS_MATCHDAY:
    raise ValueError(
        f'analysis_matchday must be from {MIN_ANALYSIS_MATCHDAY} to {MAX_ANALYSIS_MATCHDAY}; '
        'only matchdays 1 through 34 can be analysed.'
    )

next_matchday = analysis_matchday + 1 if analysis_matchday < MAX_ANALYSIS_MATCHDAY else None

teams_path = SOFASCORE_REFERENCE_DIR / 'bundesliga_teams.json'
match_ids_path = (
    SOFASCORE_MATCH_IDS_DIR / f'match_ids_{next_matchday}.json'
    if next_matchday is not None
    else None
)
execution_datetime = datetime.now().astimezone()
execution_timestamp = execution_datetime.isoformat(timespec='microseconds')
filename_timestamp = execution_datetime.strftime('%Y-%m-%d_%H-%M-%S_%f%z')
snapshot_output_path = (
    ensure_directory(SOFASCORE_UPCOMING_MATCHES_DIR)
    / (
        f'upcoming_matches_md_{next_matchday}_{filename_timestamp}.json'
        if next_matchday is not None
        else f'upcoming_matches_after_md_{analysis_matchday}_{filename_timestamp}.json'
    )
)

print(f'Squad-analysis matchday: {analysis_matchday}')
if next_matchday is None:
    print('No Bundesliga matchday follows Matchday 34; no later Bundesliga fixture will be searched.')
else:
    print(f'Following Bundesliga matchday to check: {next_matchday}')
print(f'Team input file: {teams_path}')
if match_ids_path is not None:
    print(f'Match-ID validation file: {match_ids_path}')
print(f'Snapshot output file: {snapshot_output_path}')


What Bundesliga matchday are you analysing your squad for?  1


Squad-analysis matchday: 1
Following Bundesliga matchday to check: 2
Team input file: C:\kickbase project\outputs\sofascore\reference\bundesliga_teams.json
Match-ID validation file: C:\kickbase project\outputs\sofascore\match_ids\match_ids_2.json
Snapshot output file: C:\kickbase project\outputs\sofascore\upcoming_matches\upcoming_matches_md_2_2026-08-23_00-49-34_384394+0200.json


## 3. Load and validate local inputs

The shared team reference supplies the teams to process. The derived following matchday's existing SofaScore match-ID output supplies the only event IDs eligible for `next_bundesliga_match`.

In [3]:
def load_json_file(path: Path, description: str) -> Any:
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except FileNotFoundError as exc:
        raise FileNotFoundError(f'{description} file cannot be found: {path}') from exc
    except UnicodeDecodeError as exc:
        raise ValueError(f'{description} file is not valid UTF-8: {path}') from exc
    except json.JSONDecodeError as exc:
        raise ValueError(
            f'{description} file is not valid JSON (line {exc.lineno}, column {exc.colno}): {path}'
        ) from exc
    except OSError as exc:
        raise OSError(f'Could not read {description} file {path}: {exc}') from exc


raw_teams = load_json_file(teams_path, 'Bundesliga team')
if not isinstance(raw_teams, dict):
    raise ValueError('The team JSON must be an object keyed by numeric team ID.')

teams: dict[int, dict[str, str]] = {}
for team_key, team_record in raw_teams.items():
    if not isinstance(team_record, dict):
        raise ValueError(f'Team entry {team_key!r} must be a JSON object.')
    team_id = team_record.get('team_id')
    team_name = team_record.get('team')
    if not isinstance(team_id, int) or isinstance(team_id, bool) or team_id < 1:
        raise ValueError(f'Team entry {team_key!r} has no valid positive team_id.')
    if str(team_id) != str(team_key):
        raise ValueError(f'Team key {team_key!r} does not match team_id {team_id}.')
    if not isinstance(team_name, str) or not team_name.strip():
        raise ValueError(f'Team entry {team_key!r} has no valid team name.')
    if team_id in teams:
        raise ValueError(f'Duplicate team ID in team file: {team_id}.')
    teams[team_id] = {'team': team_name.strip()}

expected_match_ids: set[int] = set()
if next_matchday is not None:
    raw_match_ids = load_json_file(match_ids_path, f'Matchday {next_matchday} match-ID')
    if not isinstance(raw_match_ids, list):
        raise ValueError('The match-ID JSON must be a list of match objects.')
    for index, match in enumerate(raw_match_ids, start=1):
        if not isinstance(match, dict):
            raise ValueError(f'Match-ID entry {index} must be a JSON object.')
        match_id = match.get('match_id')
        if not isinstance(match_id, int) or isinstance(match_id, bool) or match_id < 1:
            raise ValueError(f'Match-ID entry {index} has no valid match_id.')
        if match_id in expected_match_ids:
            raise ValueError(f'Duplicate match ID {match_id} in {match_ids_path}.')
        expected_match_ids.add(match_id)
    if not expected_match_ids:
        raise ValueError(f'Matchday {next_matchday} match-ID file contains no match IDs.')

print(f'Loaded and validated {len(teams)} unique Bundesliga team(s).')
if next_matchday is not None:
    print(f'Loaded {len(expected_match_ids)} valid SofaScore match ID(s) for matchday {next_matchday}.')


Loaded and validated 18 unique Bundesliga team(s).
Loaded 9 valid SofaScore match ID(s) for matchday 2.


## 4. SofaScore API and event helpers

These helpers retrieve API JSON through Chrome, validate the response shape, and normalize only complete upcoming events that actually involve the current team.

In [4]:
class SofaScorePageError(RuntimeError):
    """Raised when Chrome cannot return a usable SofaScore event page."""


def nested_get(mapping: Any, *keys: str) -> Any:
    current = mapping
    for key in keys:
        if not isinstance(current, dict):
            return None
        current = current.get(key)
    return current


def is_valid_int(value: Any) -> bool:
    return isinstance(value, int) and not isinstance(value, bool)


def fetch_team_events_page(driver: Any, team_id: int, page: int) -> tuple[list[dict[str, Any]], bool]:
    url = TEAM_EVENTS_URL_TEMPLATE.format(team_id=team_id, page=page)
    print(f'    Loading page {page}: {url}')
    try:
        driver.get(url)
    except TimeoutException as exc:
        raise SofaScorePageError(f'Timed out after {PAGE_LOAD_TIMEOUT_SECONDS} seconds loading {url}.') from exc
    except WebDriverException as exc:
        raise SofaScorePageError(f'Chrome could not load {url}: {exc}') from exc

    try:
        WebDriverWait(driver, WAIT_TIMEOUT_SECONDS).until(
            EC.presence_of_element_located((By.TAG_NAME, 'pre'))
        )
    except TimeoutException as exc:
        raise SofaScorePageError(f'No <pre> element appeared within {WAIT_TIMEOUT_SECONDS} seconds for {url}.') from exc
    except WebDriverException as exc:
        raise SofaScorePageError(f'Chrome could not inspect the rendered response for {url}: {exc}') from exc

    pre_tag = BeautifulSoup(driver.page_source, 'html.parser').find('pre')
    if pre_tag is None:
        raise SofaScorePageError(f'Rendered page contains no <pre> element: {url}')
    response_text = pre_tag.get_text().strip()
    if not response_text:
        raise SofaScorePageError(f'The <pre> element is empty: {url}')
    try:
        payload = json.loads(response_text)
    except json.JSONDecodeError as exc:
        raise SofaScorePageError(f'Invalid JSON for {url} (line {exc.lineno}, column {exc.colno}).') from exc
    if not isinstance(payload, dict) or not isinstance(payload.get('events'), list):
        raise SofaScorePageError(f'SofaScore response has no valid events list: {url}')

    has_next_page = payload.get('hasNextPage')
    if not isinstance(has_next_page, bool):
        print('    Warning: hasNextPage is missing or invalid; treating it as false.')
        has_next_page = False
    return payload['events'], has_next_page


def parse_upcoming_event(event: Any, team_id: int) -> dict[str, Any] | None:
    if not isinstance(event, dict):
        print('    Warning: skipped a non-object event.')
        return None
    event_id = event.get('id')
    timestamp = event.get('startTimestamp')
    home_team_id = nested_get(event, 'homeTeam', 'id')
    away_team_id = nested_get(event, 'awayTeam', 'id')
    if not is_valid_int(event_id) or event_id < 1:
        print('    Warning: skipped an event with no valid numeric ID.')
        return None
    if not isinstance(timestamp, (int, float)) or isinstance(timestamp, bool):
        print(f'    Warning: skipped event {event_id}; startTimestamp is missing or invalid.')
        return None
    if not is_valid_int(home_team_id) or not is_valid_int(away_team_id):
        print(f'    Warning: skipped event {event_id}; team IDs are missing or invalid.')
        return None
    if team_id not in (home_team_id, away_team_id):
        print(f'    Warning: skipped event {event_id}; team ID {team_id} matches neither side.')
        return None
    try:
        readable_date = datetime.fromtimestamp(timestamp, tz=timezone.utc).isoformat()
    except (OverflowError, OSError, ValueError) as exc:
        print(f'    Warning: skipped event {event_id}; invalid timestamp ({exc}).')
        return None

    if team_id == home_team_id:
        venue = 'home'
        opponent = nested_get(event, 'awayTeam', 'name')
        opponent_id = away_team_id
    else:
        venue = 'away'
        opponent = nested_get(event, 'homeTeam', 'name')
        opponent_id = home_team_id

    round_info = event.get('roundInfo')
    round_value = round_info.get('round') if isinstance(round_info, dict) else event.get('round')
    return {
        'match_id': event_id,
        'date': readable_date,
        'timestamp': timestamp,
        'competition': nested_get(event, 'tournament', 'name'),
        'opponent': opponent,
        'opponent_id': opponent_id,
        'venue': venue,
        'unique_tournament_id': nested_get(event, 'tournament', 'uniqueTournament', 'id'),
        'home_team': nested_get(event, 'homeTeam', 'name'),
        'home_team_id': home_team_id,
        'away_team': nested_get(event, 'awayTeam', 'name'),
        'away_team_id': away_team_id,
        'round': round_value,
        'season_id': nested_get(event, 'season', 'id'),
        'status_type': nested_get(event, 'status', 'type'),
    }


## 5. Collect, validate, summarize, and export

Page zero establishes the next overall event after chronological sorting. Later pages are requested only while searching for the match-ID-validated Bundesliga fixture. Individual failures leave that team's result in place and do not stop the remaining teams.

In [5]:
def collect_team_upcoming_matches(driver: Any, team_id: int, team_name: str) -> dict[str, Any]:
    candidates: list[dict[str, Any]] = []
    seen_event_ids: set[int] = set()
    eligible_bundesliga_candidates: list[dict[str, Any]] = []
    has_next_page = True

    for page in range(MAX_PAGES):
        try:
            events, has_next_page = fetch_team_events_page(driver, team_id, page)
        except SofaScorePageError as exc:
            print(f'    Warning: stopped pagination for {team_name}: {exc}')
            break
        if not events:
            print(f'    No events returned on page {page}; upcoming-event history has ended.')
            break

        for event in events:
            record = parse_upcoming_event(event, team_id)
            if record is None:
                continue
            event_id = record['match_id']
            if event_id in seen_event_ids:
                print(f'    Warning: duplicate event {event_id} across upcoming-event pages; skipped.')
                continue
            seen_event_ids.add(event_id)
            candidates.append(record)

            if (
                next_matchday is not None
                and record['unique_tournament_id'] == BUNDESLIGA_UNIQUE_TOURNAMENT_ID
            ):
                if event_id in expected_match_ids:
                    eligible_bundesliga_candidates.append(record)
                else:
                    print(
                        f'    Warning: Bundesliga event {event_id} is not in the requested '
                        f'matchday {next_matchday} match-ID file; rejecting it.'
                    )

        if next_matchday is None:
            break
        if eligible_bundesliga_candidates:
            break
        if not has_next_page:
            print(f'    Warning: pagination exhausted before finding matchday {next_matchday} for {team_name}.')
            break
    else:
        print(f'    Warning: reached MAX_PAGES={MAX_PAGES} before finding matchday {next_matchday} for {team_name}.')

    candidates.sort(key=lambda match: match['timestamp'])
    eligible_bundesliga_candidates.sort(key=lambda match: match['timestamp'])
    selected_bundesliga_match = (
        eligible_bundesliga_candidates[0] if eligible_bundesliga_candidates else None
    )
    next_match = candidates[0] if candidates else None
    if next_match is None:
        print(f'    Warning: no valid overall upcoming match found for {team_name}.')
    if next_matchday is None:
        print('    Info: Matchday 34 is the final Bundesliga matchday; no later Bundesliga fixture exists.')
    elif selected_bundesliga_match is None:
        print(f'    Warning: no valid Bundesliga matchday {next_matchday} match found for {team_name}.')

    return {
        'team': team_name,
        'next_match': next_match,
        'next_bundesliga_match': selected_bundesliga_match,
    }


def check_fixture_consistency(results: dict[int, dict[str, Any]]) -> None:
    if next_matchday is None:
        print('No following Bundesliga matchday exists; fixture consistency check is skipped.')
        return

    occurrences: dict[int, list[tuple[int, dict[str, Any]]]] = {}
    for team_id, result in results.items():
        match = result['next_bundesliga_match']
        if match is not None:
            occurrences.setdefault(match['match_id'], []).append((team_id, match))

    for match_id in expected_match_ids:
        entries = occurrences.get(match_id, [])
        if len(entries) != 2:
            print(f'Warning: fixture {match_id} was selected for {len(entries)} team(s); expected 2.')
            continue
        team_ids = {entry[0] for entry in entries}
        fixture_keys = {
            (match['home_team_id'], match['away_team_id'], match['timestamp'])
            for _, match in entries
        }
        home_team_id, away_team_id, _ = next(iter(fixture_keys))
        if len(fixture_keys) != 1 or team_ids != {home_team_id, away_team_id}:
            print(f'Warning: inconsistent fixture information encountered for match ID {match_id}.')


driver = None
upcoming_results: dict[int, dict[str, Any]] = {}
try:
    options = uc.ChromeOptions()
    options.add_argument('--disable-gpu')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    if HEADLESS:
        options.add_argument('--headless=new')
    try:
        driver = uc.Chrome(options=options, version_main=CHROME_MAJOR_VERSION, use_subprocess=True)
        driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT_SECONDS)
    except Exception as exc:
        raise RuntimeError(
            f'Could not initialize undetected Chrome with major version {CHROME_MAJOR_VERSION}. '
            f'Adjust CHROME_MAJOR_VERSION if needed. Original error: {exc}'
        ) from exc
    print(f'One reusable undetected Chrome instance is ready (headless={HEADLESS}).')

    for team_number, (team_id, team_info) in enumerate(teams.items(), start=1):
        team_name = team_info['team']
        print(f'[{team_number}/{len(teams)}] {team_name} (team_id={team_id})')
        try:
            upcoming_results[team_id] = collect_team_upcoming_matches(driver, team_id, team_name)
        except Exception as exc:
            print(f'    Unexpected team error: {type(exc).__name__}: {exc}. Continuing.')
            upcoming_results[team_id] = {
                'team': team_name,
                'next_match': None,
                'next_bundesliga_match': None,
            }

    check_fixture_consistency(upcoming_results)
    summary_rows = [
        {
            'team_id': team_id,
            'team': result['team'],
            'next_overall_competition': result['next_match']['competition'] if result['next_match'] else None,
            'next_overall_opponent': result['next_match']['opponent'] if result['next_match'] else None,
            'next_overall_date': result['next_match']['date'] if result['next_match'] else None,
            'next_overall_venue': result['next_match']['venue'] if result['next_match'] else None,
            'following_bundesliga_competition': result['next_bundesliga_match']['competition'] if result['next_bundesliga_match'] else None,
            'following_bundesliga_opponent': result['next_bundesliga_match']['opponent'] if result['next_bundesliga_match'] else None,
            'following_bundesliga_date': result['next_bundesliga_match']['date'] if result['next_bundesliga_match'] else None,
            'following_bundesliga_venue': result['next_bundesliga_match']['venue'] if result['next_bundesliga_match'] else None,
            'next_match_id': result['next_match']['match_id'] if result['next_match'] else None,
            'next_bundesliga_match_id': result['next_bundesliga_match']['match_id'] if result['next_bundesliga_match'] else None,
            'next_bundesliga_round': result['next_bundesliga_match']['round'] if result['next_bundesliga_match'] else None,
        }
        for team_id, result in upcoming_results.items()
    ]
    upcoming_summary_df = pd.DataFrame(summary_rows)
    display(upcoming_summary_df)

    snapshot = {
        'execution_timestamp': execution_timestamp,
        'analysis_matchday': analysis_matchday,
        'following_matchday': next_matchday,
        'requested_matchday': next_matchday,
        'teams': {str(team_id): result for team_id, result in upcoming_results.items()},
    }
    try:
        snapshot_output_path.write_text(
            json.dumps(snapshot, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
        )
    except OSError as exc:
        raise OSError(f'Could not save snapshot file {snapshot_output_path}: {exc}') from exc
    print(f'Saved {len(upcoming_results)} team result(s) to {snapshot_output_path}.')
finally:
    if driver is not None:
        try:
            driver.quit()
            print('Chrome driver closed.')
        except Exception as exc:
            print(f'Chrome driver shutdown warning: {exc}')


One reusable undetected Chrome instance is ready (headless=False).
[1/18] FC Bayern München (team_id=2672)
    Loading page 0: https://www.sofascore.com/api/v1/team/2672/events/next/0
[2/18] VfB Stuttgart (team_id=2677)
    Loading page 0: https://www.sofascore.com/api/v1/team/2677/events/next/0
[3/18] 1. FC Köln (team_id=2671)
    Loading page 0: https://www.sofascore.com/api/v1/team/2671/events/next/0
[4/18] TSG Hoffenheim (team_id=2569)
    Loading page 0: https://www.sofascore.com/api/v1/team/2569/events/next/0
[5/18] 1. FC Union Berlin (team_id=2547)
    Loading page 0: https://www.sofascore.com/api/v1/team/2547/events/next/0
[6/18] Eintracht Frankfurt (team_id=2674)
    Loading page 0: https://www.sofascore.com/api/v1/team/2674/events/next/0
[7/18] 1. FSV Mainz 05 (team_id=2556)
    Loading page 0: https://www.sofascore.com/api/v1/team/2556/events/next/0
[8/18] SC Paderborn 07 (team_id=2561)
    Loading page 0: https://www.sofascore.com/api/v1/team/2561/events/next/0
[9/18] RB Le

,team_id,team,next_overall_competition,next_overall_opponent,next_overall_date,next_overall_venue,following_bundesliga_competition,following_bundesliga_opponent,following_bundesliga_date,following_bundesliga_venue,next_match_id,next_bundesliga_match_id,next_bundesliga_round
0,2672,FC Bayern München,Bundesliga,VfB Stuttgart,2026-08-28T18:30:00+00:00,home,Bundesliga,FC Schalke 04,2026-09-05T16:30:00+00:00,away,16434087,16434032,2
1,2677,VfB Stuttgart,Bundesliga,FC Bayern München,2026-08-28T18:30:00+00:00,away,Bundesliga,1. FC Köln,2026-09-04T18:30:00+00:00,home,16434087,16434036,2
2,2671,1. FC Köln,DFB Pokal,FC Würzburger Kickers,2026-08-24T16:00:00+00:00,away,Bundesliga,VfB Stuttgart,2026-09-04T18:30:00+00:00,away,16287059,16434036,2
3,2569,TSG Hoffenheim,Bundesliga,1. FC Köln,2026-08-29T13:30:00+00:00,away,Bundesliga,Borussia Dortmund,2026-09-05T13:30:00+00:00,home,16434022,16434019,2
4,2547,1. FC Union Berlin,DFB Pokal,Eintracht Braunschweig,2026-08-23T13:30:00+00:00,away,Bundesliga,Bayer 04 Leverkusen,2026-09-05T13:30:00+00:00,away,16287045,16434042,2
5,2674,Eintracht Frankfurt,Bundesliga,1. FC Union Berlin,2026-08-29T13:30:00+00:00,away,Bundesliga,FC Augsburg,2026-09-06T15:30:00+00:00,home,16434026,16434021,2
6,2556,1. FSV Mainz 05,DFB Pokal,VfB 1921 Krieschow,2026-08-23T13:30:00+00:00,away,Bundesliga,Hamburger SV,2026-09-06T13:30:00+00:00,away,16287042,16434030,2
7,2561,SC Paderborn 07,DFB Pokal,1.FC Phönix Lübeck,2026-08-23T16:00:00+00:00,away,Bundesliga,SC Freiburg,2026-09-05T13:30:00+00:00,home,16287053,16434024,2
8,36360,RB Leipzig,Bundesliga,Borussia M'gladbach,2026-08-29T13:30:00+00:00,home,Bundesliga,SV Werder Bremen,2026-09-05T13:30:00+00:00,away,16434023,16434101,2
9,2527,Borussia M'gladbach,DFB Pokal,TSV Schott Mainz,2026-08-23T13:30:00+00:00,away,Bundesliga,SV 07 Elversberg,2026-09-05T13:30:00+00:00,home,16287039,16434028,2


Saved 18 team result(s) to C:\kickbase project\outputs\sofascore\upcoming_matches\upcoming_matches_md_2_2026-08-23_00-49-34_384394+0200.json.
Chrome driver closed.


In [ ]:
from project_paths import prune_timestamped_outputs

removed_outputs = prune_timestamped_outputs()
print(f"Pruned {len(removed_outputs)} expired timestamped output(s).")
